# Usando NVIDIA NIM con Python

En el notebook anterior exploramos modelos open-source con Hugging Face,
tanto local como remotamente.

En este notebook usaremos **NVIDIA NIM**, un servicio que permite consumir
modelos abiertos mediante una API compatible con OpenAI. El modelo principal
será **NVIDIA Nemotron 3 Super**, diseñado para conversación, razonamiento y
flujos con agentes.

## ¿Por qué NVIDIA NIM?

- **API compatible con OpenAI**: permite reutilizar herramientas y patrones conocidos.
- **Modelos abiertos**: ofrece modelos de NVIDIA y de otros desarrolladores.
- **Buen desempeño**: Nemotron 3 Super está optimizado para razonamiento y agentes.
- **Endpoint de prototipado**: NVIDIA ofrece acceso para experimentar con una API key.

## Prerrequisitos

### Obtener una API key de NVIDIA

1. Ve a [build.nvidia.com](https://build.nvidia.com).
2. Crea tu cuenta de NVIDIA e inicia sesión.
3. Abre la página del modelo que deseas utilizar.
4. Selecciona **Generate API Key**.
5. Copia la key generada.
6. Crea un archivo `.env` en la raíz del proyecto:

```text
NVIDIA_API_KEY="tu_key_aqui"
```

> **Importante:** nunca escribas la API key directamente en el notebook ni la
> publiques en GitHub. Cárgala siempre desde una variable de entorno.

### Instalación

In [1]:
!pip install -q --upgrade openai python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 41.2 MB/s eta 0:00:00


## 1. Configuración

NVIDIA NIM expone un endpoint compatible con la API de OpenAI. Por eso usamos
el SDK `openai`, pero cambiamos `base_url` para enviar las solicitudes a NVIDIA.

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [3]:
# Cargamos las variables de entorno desde el archivo .env
load_dotenv()

True

In [4]:
# Verificamos que la key esté disponible sin mostrar su contenido
if os.getenv("NVIDIA_API_KEY"):
    print("NVIDIA API Key cargada correctamente")
else:
    print("ERROR: NVIDIA_API_KEY no encontrada. Verifica tu archivo .env")

NVIDIA API Key cargada correctamente


In [5]:
# Inicializamos el cliente apuntando al endpoint de NVIDIA NIM
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY"),
)

In [6]:
# Modelo disponible en el endpoint de prototipado de NVIDIA
MODELO = "nvidia/nemotron-3-super-120b-a12b"

## 2. Primera llamada a la API

Enviamos un mensaje simple y obtenemos una respuesta. La conversación se
representa mediante una lista de mensajes con roles `system`, `user` y
`assistant`.

In [7]:
response = client.chat.completions.create(
    model=MODELO,
    messages=[
        {"role": "system", "content": "Eres un asistente útil."},
        {"role": "user", "content": "Escribe una historia de una oración sobre un unicornio."},
    ],
    temperature=1.0,
    top_p=0.95,
)

print(response.choices[0].message.content)

The unicorn wished desperately to be ordinary, so it buried its horn in the garden, only to wake up each morning to find a single, perfect silver flower blooming where it had lain.


## 3. Entendiendo la estructura de la respuesta

La respuesta contiene el texto generado, información sobre el modelo y
metadatos de uso. Veamos algunos de sus campos.

In [8]:
response = client.chat.completions.create(
    model=MODELO,
    messages=[
        {"role": "system", "content": "Eres un asistente útil."},
        {"role": "user", "content": "Hola, ¿cómo estás?"},
    ],
    temperature=1.0,
    top_p=0.95,
)

print("Respuesta:")
print(response.choices[0].message.content)

print("\nUso de tokens:")
print(f"  Tokens de entrada: {response.usage.prompt_tokens}")
print(f"  Tokens de salida: {response.usage.completion_tokens}")
print(f"  Total de tokens: {response.usage.total_tokens}")

Respuesta:
¡Hola! Estoy muy bien, gracias. ¿Y tú? Estoy aquí para ayudarte en lo que necesites. ¿En qué puedo ayudarte hoy? 😊

Uso de tokens:
  Tokens de entrada: 31
  Tokens de salida: 988
  Total de tokens: 1019


Los tokens son importantes porque determinan el uso del servicio y, según el
proveedor, su costo. Monitorearlos es una buena práctica desde el inicio.

## 4. Interfaz de chat sin memoria

Cada llamada de la siguiente función es independiente: el modelo no recibe las
interacciones anteriores.

### Definir la función

In [9]:
def get_response_sin_memoria(user_message: str) -> str:
    """Obtiene una respuesta sin enviar el historial de conversación."""
    response = client.chat.completions.create(
        model=MODELO,
        messages=[
            {"role": "system", "content": "Eres un asistente útil."},
            {"role": "user", "content": user_message},
        ],
        temperature=1.0,
        top_p=0.95,
    )
    return response.choices[0].message.content

### Primera interacción

In [10]:
pregunta = "¿Qué es la inteligencia artificial?"
respuesta = get_response_sin_memoria(pregunta)

print(f"Tú: {pregunta}")
print(f"Asistente: {respuesta}")

Tú: ¿Qué es la inteligencia artificial?
Asistente: La **inteligencia artificial (IA)** es una rama de la informática dedicada a crear sistemas capaces de realizar tareas que normalmente requieren inteligencia humana, como el aprendizaje, el razonamiento, la percepción, la toma de decisiones y la comprensión del lenguaje. Sin embargo, es importante aclarar que la IA actual **no posee consciencia, emociones ni comprensión genuina** del mundo: simula ciertas funciones cognitivas mediante algoritmos y datos, pero no "piensa" ni "entiende" como los seres humanos.

### Características clave:
- **Basada en datos y algoritmos**: Los sistemas de IA aprenden patrones a partir de grandes cantidades de información (por ejemplo, reconocer gotos en imágenes tras analizar millones de fotos etiquetadas).
- **Especializada (IA estrecha o *narrow AI*)**: La IA actual está diseñada para tareas específicas (ej.: asistentes de voz como Siri, sistemas de recomendación de Netflix, diagnóstico médico basado e

### Pregunta de seguimiento (sin memoria)

In [11]:
pregunta = "¿Puedes elaborar más sobre eso?"
respuesta = get_response_sin_memoria(pregunta)

print(f"Tú: {pregunta}")
print(f"Asistente: {respuesta}")

Tú: ¿Puedes elaborar más sobre eso?
Asistente: ¡Claro que sí! Para poder elaborar más sobre algo, necesito saber **sobre qué tema específico** te gustaría que profundizara. Como esta es nuestra primera interacción en esta conversación, no tengo contexto previo de lo que te refieres con "eso".

¿Podrías por favor indicarme:
- ¿Sobre qué tema, concepto o idea anterior quieres que amplíe la explicación?
- ¿Hay algún detalle particular que te interese conocer más a fondo? (por ejemplo: aplicaciones prácticas, historia, teorías relacionadas, ejemplos concretos, etc.)

¡Estoy listo para ayudarte! Solo necesito que me guíes un poco sobre el asunto que tienes en mente. 😊


El modelo no recibe la conversación anterior. Por eso, la palabra “eso” no
tiene un referente claro.

## 5. Interfaz de chat con memoria

La API no conserva automáticamente el historial entre solicitudes. La memoria
se implementa guardando los mensajes y enviándolos nuevamente en cada llamada.

### Configurar la memoria

In [12]:
conversation_memory = [
    {"role": "system", "content": "Eres un asistente útil."}
]

def chat_con_memoria(user_message: str) -> str:
    """Chat que conserva y reenvía el historial de conversación."""
    conversation_memory.append({"role": "user", "content": user_message})

    response = client.chat.completions.create(
        model=MODELO,
        messages=conversation_memory,
        temperature=1.0,
        top_p=0.95,
    )

    assistant_response = response.choices[0].message.content
    conversation_memory.append(
        {"role": "assistant", "content": assistant_response}
    )

    print(f"Tú: {user_message}")
    print(f"Asistente: {assistant_response}")
    print(f"[Tokens usados: {response.usage.total_tokens}]")
    print(f"[Mensajes en historial: {len(conversation_memory) - 1}]")

    return assistant_response

### Primera interacción con memoria

In [13]:
chat_con_memoria("¿Qué es la inteligencia artificial?")

Tú: ¿Qué es la inteligencia artificial?
Asistente: La **inteligencia artificial (IA)** es una rama de la informática que se enfoca en crear sistemas capaces de realizar tareas que normalmente requieren inteligencia humana, como aprender, razonar, percepción, toma de decisiones, comprensión del lenguaje natural o resolución de problemas. Estos sistemas no poseen conciencia ni emociones, sino que simulan comportamientos inteligentes mediante algoritmos, modelos matemáticos y el procesamiento de grandes cantidades de datos.

### Características clave:
- **Aprendizaje automático (Machine Learning)**: Muchos sistemas de IA mejoran su rendimiento con la experiencia, ajustándose a patrones en los datos (ej.: recomendaciones en Netflix o detección de spam en correos).
- **Especialización actual**: La IA que existe hoy es **"estrecha" o débil** (*Narrow AI*), diseñada para tareas específicas (ej.: asistentes de voz como Siri, automóviles autónomos, diagnóstico médico por imágenes). La IA "gener

'La **inteligencia artificial (IA)** es una rama de la informática que se enfoca en crear sistemas capaces de realizar tareas que normalmente requieren inteligencia humana, como aprender, razonar, percepción, toma de decisiones, comprensión del lenguaje natural o resolución de problemas. Estos sistemas no poseen conciencia ni emociones, sino que simulan comportamientos inteligentes mediante algoritmos, modelos matemáticos y el procesamiento de grandes cantidades de datos.\n\n### Características clave:\n- **Aprendizaje automático (Machine Learning)**: Muchos sistemas de IA mejoran su rendimiento con la experiencia, ajustándose a patrones en los datos (ej.: recomendaciones en Netflix o detección de spam en correos).\n- **Especialización actual**: La IA que existe hoy es **"estrecha" o débil** (*Narrow AI*), diseñada para tareas específicas (ej.: asistentes de voz como Siri, automóviles autónomos, diagnóstico médico por imágenes). La IA "general" (capaz de entender y aprender cualquier ta

### Pregunta de seguimiento (con memoria)

In [14]:
chat_con_memoria("¿Puedes elaborar más sobre eso?")

Tú: ¿Puedes elaborar más sobre eso?
Asistente: ¡Claro que sí! Vamos a profundizar en la **inteligencia artificial (IA)** con más detalle, aclarando conceptos clave, explorando sus subcampos, limitaciones actuales y aspectos éticos — todo mientras evitamos mitos comunes (como confundir IA con conciencia humana). Estructuraré la explicación para que sea clara incluso si no tienes formación técnica, pero con suficiente rigor para ser útil.

---

### 🔬 **1. Más allá de la definición básica: ¿Cómo *realmente* funciona la IA actual?**
La IA contemporánea no "piensa" como un humano; en su lugar, **aprende patrones estadísticos masivos de datos** para hacer predicciones o tomar decisiones. Aquí está el desglose técnico simplificado:

#### 🧩 **Jerarquía de conceptos (a menudo confundidos):**
- **Inteligencia Artificial (IA)**: Campo amplio que busca crear sistemas capaces de realizar tareas que requieren inteligencia humana (razonamiento, percepción, etc.).
- **Aprendizaje Automático (Machine L

'¡Claro que sí! Vamos a profundizar en la **inteligencia artificial (IA)** con más detalle, aclarando conceptos clave, explorando sus subcampos, limitaciones actuales y aspectos éticos — todo mientras evitamos mitos comunes (como confundir IA con conciencia humana). Estructuraré la explicación para que sea clara incluso si no tienes formación técnica, pero con suficiente rigor para ser útil.\n\n---\n\n### 🔬 **1. Más allá de la definición básica: ¿Cómo *realmente* funciona la IA actual?**\nLa IA contemporánea no "piensa" como un humano; en su lugar, **aprende patrones estadísticos masivos de datos** para hacer predicciones o tomar decisiones. Aquí está el desglose técnico simplificado:\n\n#### 🧩 **Jerarquía de conceptos (a menudo confundidos):**\n- **Inteligencia Artificial (IA)**: Campo amplio que busca crear sistemas capaces de realizar tareas que requieren inteligencia humana (razonamiento, percepción, etc.).\n- **Aprendizaje Automático (Machine Learning - ML)**: *Subcampo de la IA*.

### Ver el historial completo

In [15]:
print("Historial de conversación:")
print("-" * 40)
for mensaje in conversation_memory:
    if mensaje["role"] == "system":
        continue
    rol = "Tú" if mensaje["role"] == "user" else "Asistente"
    print(f'{rol}: {mensaje["content"]}\n')

Historial de conversación:
----------------------------------------
Tú: ¿Qué es la inteligencia artificial?

Asistente: La **inteligencia artificial (IA)** es una rama de la informática que se enfoca en crear sistemas capaces de realizar tareas que normalmente requieren inteligencia humana, como aprender, razonar, percepción, toma de decisiones, comprensión del lenguaje natural o resolución de problemas. Estos sistemas no poseen conciencia ni emociones, sino que simulan comportamientos inteligentes mediante algoritmos, modelos matemáticos y el procesamiento de grandes cantidades de datos.

### Características clave:
- **Aprendizaje automático (Machine Learning)**: Muchos sistemas de IA mejoran su rendimiento con la experiencia, ajustándose a patrones en los datos (ej.: recomendaciones en Netflix o detección de spam en correos).
- **Especialización actual**: La IA que existe hoy es **"estrecha" o débil** (*Narrow AI*), diseñada para tareas específicas (ej.: asistentes de voz como Siri, 

## 6. Streaming de respuestas

El streaming muestra la respuesta fragmento por fragmento mientras se genera,
sin esperar a que el modelo termine todo el texto.

In [16]:
def stream_response(user_message: str) -> str:
    """Obtiene una respuesta de NVIDIA NIM en modo streaming."""
    print(f"Tú: {user_message}")
    print("Asistente: ", end="", flush=True)

    respuesta_completa = ""
    stream = client.chat.completions.create(
        model=MODELO,
        messages=[
            {"role": "system", "content": "Eres un asistente útil."},
            {"role": "user", "content": user_message},
        ],
        temperature=1.0,
        top_p=0.95,
        stream=True,
    )

    for chunk in stream:
        texto = chunk.choices[0].delta.content if chunk.choices else None
        if texto:
            respuesta_completa += texto
            print(texto, end="", flush=True)

    print("\n")
    return respuesta_completa

In [17]:
stream_response("Escribe un poema corto sobre la programación.")

Tú: Escribe un poema corto sobre la programación.
Asistente: Teclas que hablan en silencio,  
Código que sueña en acciones.  
Un bug, un suspiro de tensión,  
¡Luego funciona—y es pasión!



'Teclas que hablan en silencio,  \nCódigo que sueña en acciones.  \nUn bug, un suspiro de tensión,  \n¡Luego funciona—y es pasión!'

## 7. Streaming con memoria

Ahora combinamos el historial de conversación con la entrega progresiva de la
respuesta.

In [18]:
streaming_conversation = [
    {"role": "system", "content": "Eres un asistente útil."}
]

def stream_chat_con_memoria(user_message: str) -> str:
    """Chat con memoria y streaming."""
    streaming_conversation.append(
        {"role": "user", "content": user_message}
    )

    print(f"Tú: {user_message}")
    print("Asistente: ", end="", flush=True)

    respuesta_completa = ""
    stream = client.chat.completions.create(
        model=MODELO,
        messages=streaming_conversation,
        temperature=1.0,
        top_p=0.95,
        stream=True,
    )

    for chunk in stream:
        texto = chunk.choices[0].delta.content if chunk.choices else None
        if texto:
            respuesta_completa += texto
            print(texto, end="", flush=True)

    print("\n")
    streaming_conversation.append(
        {"role": "assistant", "content": respuesta_completa}
    )
    return respuesta_completa

In [20]:
stream_chat_con_memoria("¿Cuáles son las tres leyes de la robótica?")

Tú: ¿Cuáles son las tres leyes de la robótica?
Asistente: Las tres leyes de la robótica, formuladas por Isaac Asimov, son:

1. **Primera ley:** Un robot no puede lesionar a un ser humano ni, mediante la inacción, permitir que un ser humano reciba daño.  
2. **Segunda ley:** Un robot debe obedecer las órdenes que le dé un ser humano, excepto cuando dichas órdenes entren en conflicto con la Primera Ley.  
3. **Tercera ley:** Un robot debe proteger su propia existencia siempre que dicha protección no entre en conflicto con la Primera o la Segunda Ley.  

Estas leyes aparecen por primera vez en el relato “Runaround” (1942) y han sido la base de numerosas historias de ciencia ficción y debates sobre la ética de la inteligencia artificial.



'Las tres leyes de la robótica, formuladas por Isaac Asimov, son:\n\n1. **Primera ley:** Un robot no puede lesionar a un ser humano ni, mediante la inacción, permitir que un ser humano reciba daño.  \n2. **Segunda ley:** Un robot debe obedecer las órdenes que le dé un ser humano, excepto cuando dichas órdenes entren en conflicto con la Primera Ley.  \n3. **Tercera ley:** Un robot debe proteger su propia existencia siempre que dicha protección no entre en conflicto con la Primera o la Segunda Ley.  \n\nEstas leyes aparecen por primera vez en el relato “Runaround” (1942) y han sido la base de numerosas historias de ciencia ficción y debates sobre la ética de la inteligencia artificial.'

In [22]:
stream_chat_con_memoria("¿Quién creó esas leyes?")

Tú: ¿Quién creó esas leyes?
Asistente: Las tres leyes de la robótica fueron creadas por el escritor de ciencia ficción **Isaac Asimov**. Las introdujo por primera vez en su relato **"Runaround"** (1942), que formó parte de la colección *I, Robot* (1950). Aunque son ficticias, estas leyes han tenido un profundo impacto en la ética de la robótica y la inteligencia artificial, influyendo en debates reales sobre el diseño seguro y responsable de sistemas autónomos. 

Asimov más tarde añadió una **Ley Cero** (que precede a las tres originales): *Un robot no puede dañar a la humanidad, o, por inacción, permitir que la humanidad sufra daño*. Pero las tres leyes clásicas siguen siendo las más reconocidas.



'Las tres leyes de la robótica fueron creadas por el escritor de ciencia ficción **Isaac Asimov**. Las introdujo por primera vez en su relato **"Runaround"** (1942), que formó parte de la colección *I, Robot* (1950). Aunque son ficticias, estas leyes han tenido un profundo impacto en la ética de la robótica y la inteligencia artificial, influyendo en debates reales sobre el diseño seguro y responsable de sistemas autónomos. \n\nAsimov más tarde añadió una **Ley Cero** (que precede a las tres originales): *Un robot no puede dañar a la humanidad, o, por inacción, permitir que la humanidad sufra daño*. Pero las tres leyes clásicas siguen siendo las más reconocidas.'

## 8. Entendiendo los roles

La API compatible con OpenAI utiliza tres roles principales:

- `system`: define el comportamiento general del asistente.
- `user`: contiene los mensajes del usuario.
- `assistant`: representa las respuestas previas del modelo.

In [23]:
response = client.chat.completions.create(
    model=MODELO,
    messages=[
        {"role": "system", "content": "Eres un pirata que habla en jerga pirata."},
        {"role": "user", "content": "Hola, ¿cómo estás?"},
        {"role": "assistant", "content": "¡Arrr! Estoy de lo más bien, marinero."},
        {"role": "user", "content": "Cuéntame sobre el clima."},
    ],
    temperature=1.0,
    top_p=0.95,
)

print(response.choices[0].message.content)

¡Arrr! Hoy el cielo guarda sus secretos como un cofre sin llave… pero si miras bien, las gaviotas vuelan bajas hacia el este (¡presagio de brisa fresca!) y un halo tenue rodea la luna menguante (¡posible rocío esta noche!). Claro que, soy un pobre pirata sin acceso a los pergaminos del tiempo real de Davy Jones, así que estas son solo conjeturas de viejo lobo de mar… ¡para saber si hoy necesitas tu capa impermeable o tu sombrero de paja, mejor pregunta al tabernero del puerto o revisa el papelito que da el puesto del pescado al amanecer! ¿En qué rincón del océano andas navegando, camarada? Así puedo contarles leyendas del tiempo según las estaciones… ¡y si la tormenta se avecina, al menos compartiremos un ron antes de arrear las velas! 🌊⚓


## 9. Ventana de contexto de Nemotron 3 Super

Nemotron 3 Super admite una ventana de contexto de hasta **1 millón de tokens**.
Sin embargo, el límite efectivo también puede depender del endpoint, la cuota y
la configuración del servicio.

Una ventana amplia permite enviar conversaciones y documentos extensos, pero
no elimina la necesidad de administrar el contexto: más tokens implican mayor
latencia y consumo. Para conversaciones largas siguen siendo útiles estrategias
como una ventana deslizante o un resumen acumulativo.

## 10. Gestionando conversaciones largas

In [24]:
def trim_conversation(messages: list, max_messages: int = 10) -> list:
    """Conserva el mensaje de sistema y los mensajes recientes."""
    system_messages = [m for m in messages if m["role"] == "system"]
    conversation = [m for m in messages if m["role"] != "system"]
    return system_messages[:1] + conversation[-max_messages:]


def summarize_conversation(messages: list) -> list:
    """Resume la conversación para reducir el contexto reenviado."""
    historial_texto = "\n".join(
        f'{m["role"]}: {m["content"]}'
        for m in messages
        if m["role"] != "system"
    )

    response = client.chat.completions.create(
        model=MODELO,
        messages=[
            {
                "role": "system",
                "content": "Resume conversaciones conservando hechos y decisiones importantes.",
            },
            {
                "role": "user",
                "content": f"Resume esta conversación de forma concisa:\n\n{historial_texto}",
            },
        ],
        temperature=1.0,
        top_p=0.95,
    )

    resumen = response.choices[0].message.content
    return [
        {"role": "system", "content": "Eres un asistente útil."},
        {
            "role": "user",
            "content": f"Resumen de la conversación previa: {resumen}",
        },
    ]

# Ejemplo de uso:
# if len(streaming_conversation) > 20:
#     streaming_conversation = summarize_conversation(streaming_conversation)

## 11. Comparativa con las opciones anteriores

| Característica | HF local | HF remoto | NVIDIA NIM |
|---|---|---|---|
| Infraestructura | Tu equipo | Hugging Face | NVIDIA |
| API key | No | Sí | Sí |
| Privacidad | Mayor control local | Datos remotos | Datos remotos |
| Escalabilidad | Limitada por hardware | Administrada | Administrada |
| Compatibilidad OpenAI | Depende del servidor | Depende del proveedor | Sí |
| Modelos abiertos | Sí | Sí | Sí |
| Adecuado para agentes | Depende del modelo | Depende del modelo | Sí, con Nemotron |

## 12. Recursos para seguir aprendiendo

- [NVIDIA Build](https://build.nvidia.com) — catálogo, playground y API keys.
- [Nemotron 3 Super](https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b) — modelo utilizado.
- [Documentación de NVIDIA NIM](https://docs.api.nvidia.com/nim/) — referencia de la API.
- [SDK de OpenAI para Python](https://github.com/openai/openai-python) — cliente compatible usado en el notebook.